# 19 — Capstone: the whole desk, end to end

**What you'll learn**

- The full pipeline as a single run: the desk's read tools reach the agent over **MCP** (`MCPBridge`), a supplier peer is consulted over **A2A** (`ask_supplier`), an agent composed from the parts you built clears the queue, and every step is traced
- Assembling that agent by composition — `run_agent` + a per-ticket `Budget` + a `require_approval` gate + the `check_decision` verifier + a `Tracer` and Phoenix — parts you built across chapters 02–14
- Scoring the run against the gold **TEST** split with `score_ticket`: the eight tickets the course has never trained or tuned on, an honest held-out evaluation
- Reading the outcome — aggregate accuracy, the measured cost from `usage_summary`, and the span tree for one ticket
- Tearing both servers down in-notebook, leaving no orphan process and no bound port

*Time: ~4 min on a first live run; a few seconds cached. Cost: ~$0.02 (eight live triages over MCP + A2A). Cached reruns are free.*

> **Before running this notebook:** `pip install -e ".[mcp,a2a]"` (once). It pulls in the official MCP Python SDK (`mcp>=1.28,<2`) and the `a2a-sdk` plus a small ASGI stack (`uvicorn`, `starlette`) — the shop-desk MCP server and the supplier peer both run on localhost. Everything else stays the same.

## The brief: the whole desk, for real

Eighteen chapters in, the ops desk has a full drawer of parts, and Part 5 held each one up against the frameworks that package it. The finale spends them all at once, on the daily job the course has circled since chapter 00: **clear the morning queue.** A handful of return tickets came in overnight; for each, the agent looks up the order and the customer, finds the governing policy, computes the amount, and carries out the decision — inside a budget, never moving money a rule would not, leaving a trace we can read afterward.

What is new is that nothing here is a stand-in. The read tools are not local functions this time — they arrive over **MCP**, from the server you built in chapter 13. The one fact Larkspur cannot look up, a supplier's restock timeline, comes from **another organization's agent over A2A**, the peer from chapter 14. The loop, the budget, the approval gate, the verifier, the tracer are the exact objects you built in Parts 1 and 2. The capstone is not new machinery; it is the machinery, wired together and turned on.

And we grade it honestly. The eight tickets we score against are the **TEST** split — held out from the start. No chapter fit a prompt, tuned a rule, or calibrated a threshold against them (chapter 04 worked the dev split; the rules were authored against `docs/WORLD.md`). So the number at the end is the one that counts: how the assembled desk does on tickets it has never, in any chapter, been shown.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The queue and the answer key

Load the eight TEST tickets, the orders and customers they reference, a renderer that turns a ticket into the desk's prompt, and a `Tracer` (chapter 03) to wrap the whole run. Each ticket carries its `gold` label — the decision `shoplab.rules.decide` computed — which is the answer key we hold back until the end.

In [ ]:
import json
from shoplab.trace import Tracer, export_phoenix
from shoplab.world import load_tickets, load_orders, load_customers

tracer = Tracer()                                    # ch03 structural spans (triage + tools)
orders = {o["order_id"]: o for o in load_orders()}
customers = {c["customer_id"]: c for c in load_customers()}
TEST = load_tickets()["test"]                        # 8 gold tickets, never trained/tuned on

def render(t):
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']} qty {t['qty']}, condition "
            f"{t['item_condition']}, {t['days_since_delivery']} days since delivery, "
            f"photo evidence {t['evidence_photo']}, requested {t['requested_action']}. "
            f"Customer writes: {t['reason_text']}")

for t in TEST:
    g = t["gold"]
    print(f"{t['ticket_id']}  gold={g['decision']:15} {g['policy_id']:16} refund={g['refund_usd']}")

> **What you should see:** eight tickets, `TKT-2233`–`TKT-2240`, each printed with the gold decision the rules engine assigned — the answer key. The spread covers the range, including at least one `partial_refund` and one `escalate`. These are the held-out split: no chapter fit anything to them, so the score at the end is the honest one.

## Read tools, over MCP

The desk's four read tools — `get_order`, `get_customer`, `search_policy`, `check_inventory` — are not local functions here. They live behind the MCP server `mcp_servers/shopdesk_server.py` (chapter 13) and reach the agent over a stdio transport. MCP servers expose exactly these primitives — [tools the model may call, resources, and prompts](https://modelcontextprotocol.io/specification/2026-07-28) — and the course pins `mcp>=1.28,<2` because `pip install mcp` now installs the reworked [2.x line](https://github.com/modelcontextprotocol/python-sdk) that renamed the server class.

The bridge is chapter 13's `MCPBridge`, unchanged: a background-thread event loop keeps one live `ClientSession` open, and `as_tools()` wraps each MCP tool in a `shoplab.tools.Tool` whose `fn` round-trips over the wire. `run_agent` will call these exactly as it called local tools — the loop does not change, only the transport does.

In [ ]:
import asyncio, sys, threading
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = Path("mcp_servers/shopdesk_server.py").resolve()

class MCPBridge:
    """Owns a background-thread event loop running one live MCP stdio session."""

    def __init__(self, server_path, python=sys.executable, timeout=10.0):
        self.params = StdioServerParameters(
            command=python, args=[str(server_path)], env=os.environ.copy())
        self.timeout = timeout
        self._loop = asyncio.new_event_loop()
        self._ready, self._stop = threading.Event(), threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._session, self.mcp_tools, self._exc = None, [], None

    def _run(self):
        asyncio.set_event_loop(self._loop)
        self._loop.run_until_complete(self._serve())

    async def _serve(self):
        try:
            async with stdio_client(self.params) as (read, write):
                async with ClientSession(read, write) as session:
                    await asyncio.wait_for(session.initialize(), timeout=self.timeout)
                    listed = await asyncio.wait_for(session.list_tools(), timeout=self.timeout)
                    self._session, self.mcp_tools = session, listed.tools
                    self._ready.set()
                    while not self._stop.is_set():          # keep the session alive
                        await asyncio.sleep(0.05)
        except Exception as e:                              # surface a startup failure
            self._exc = e
            self._ready.set()

    def __enter__(self):
        self._thread.start()
        if not self._ready.wait(timeout=self.timeout + 5):
            raise RuntimeError("MCP bridge did not become ready")
        if self._exc:
            raise self._exc
        return self

    def __exit__(self, *exc):
        self._stop.set()                                   # let _serve unwind the contexts
        self._thread.join(timeout=self.timeout + 5)        # subprocess reaped here
        self._loop.close()

    def _call_sync(self, name, args):
        fut = asyncio.run_coroutine_threadsafe(
            self._session.call_tool(name, args), self._loop)
        return fut.result(timeout=self.timeout)

    def _unwrap(self, res):
        """A CallToolResult -> a plain JSON-able value run_tool can serialise."""
        if res.structuredContent is not None:
            sc = res.structuredContent
            if isinstance(sc, dict) and set(sc) == {"result"}:   # FastMCP list wrap
                return sc["result"]
            return sc
        if res.content and getattr(res.content[0], "text", None) is not None:
            txt = res.content[0].text
            if res.isError:
                return {"error": txt}
            try:
                return json.loads(txt)
            except ValueError:
                return txt
        return {"error": "empty MCP result"} if res.isError else None

    def as_tools(self):
        """dict[name -> shoplab Tool] whose fn round-trips through the session."""
        from shoplab.tools import Tool
        tools = {}
        for mt in self.mcp_tools:
            def make(n):
                def fn(**args):
                    return self._unwrap(self._call_sync(n, args))
                return fn
            tools[mt.name] = Tool(name=mt.name, description=mt.description or "",
                                  params=mt.inputSchema, fn=make(mt.name))
        return tools

In [ ]:
mcp_bridge = MCPBridge(SERVER)
mcp_bridge.__enter__()                                      # server subprocess up; torn down at the end
mcp_tools = mcp_bridge.as_tools()
print("read tools, now over MCP:", sorted(mcp_tools))
print("get_order(ORD-7301) status:", mcp_tools["get_order"].fn(order_id="ORD-7301")["status"])

> **What you should see:** five tools resolved over the stdio transport — `check_inventory`, `get_customer`, `get_order`, `refund_preview`, `search_policy` — and a live `get_order` call returning `delivered` from the server *subprocess*, not a local dict. The agent will use four of them; `refund_preview` stays unused here (it is chapter 13's `isError` demo).

## The supplier peer, over A2A

The one thing Larkspur cannot look up locally is a supplier's restock timeline. Northwind Supply is a separate organization running its own agent (chapter 14), reachable only over **A2A**. `A2ABridge` starts that supplier in a uvicorn daemon thread and keeps an A2A streaming client on a background loop, so a *synchronous* `ask_supplier(sku)` fits the agent loop like any other tool.

It sends the SKU with no region, catches the peer's [`input-required`](https://a2a-protocol.org/v1.0.0/specification/) pause, and answers with a default region to drive the task to `completed` — the multi-turn flow from chapter 14, folded into one call. We wrap it as a `Tool` so the model itself decides when to reach for the peer, rather than making the call for it.

In [ ]:
import httpx
from a2a_servers.supplier_agent import serve_in_background
from a2a.client import A2ACardResolver, ClientFactory, ClientConfig
from a2a.helpers import new_text_message, get_stream_response_text
from a2a.types import SendMessageRequest, TaskState, Role

class A2ABridge:
    """Supplier uvicorn server + a background-loop A2A streaming client. ``ask_supplier``
    sends the SKU with no region, then answers the peer's input-required question with
    ``default_region`` -- a sync call the agent loop can use like any other tool."""

    def __init__(self, default_region="west", timeout=15.0):
        self.default_region = default_region
        self.timeout = timeout
        self._loop = asyncio.new_event_loop()
        self._ready, self._stop, self._exc = threading.Event(), threading.Event(), None
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._supplier_cm = None
        self.base_url = self.card = self._hx = self._client = None

    def _run(self):
        asyncio.set_event_loop(self._loop)
        self._loop.run_until_complete(self._serve())

    async def _serve(self):
        try:
            self._supplier_cm = serve_in_background()       # uvicorn daemon thread
            self.base_url = self._supplier_cm.__enter__()   # blocks until started
            self._hx = httpx.AsyncClient(timeout=self.timeout)
            self.card = await A2ACardResolver(
                httpx_client=self._hx, base_url=self.base_url).get_agent_card()
            self._client = ClientFactory(
                ClientConfig(httpx_client=self._hx, streaming=True)).create(self.card)
            self._ready.set()
            while not self._stop.is_set():
                await asyncio.sleep(0.05)
        except Exception as e:
            self._exc = e
            self._ready.set()
        finally:
            if self._hx is not None:
                await self._hx.aclose()
            if self._supplier_cm is not None:
                self._supplier_cm.__exit__(None, None, None)

    def __enter__(self):
        self._thread.start()
        if not self._ready.wait(timeout=self.timeout + 10):
            raise RuntimeError("A2A bridge did not become ready")
        if self._exc:
            raise self._exc
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join(timeout=self.timeout + 10)
        self._loop.close()

    async def _send_one(self, text, task_id=None, context_id=None):
        msg = new_text_message(text, role=Role.ROLE_USER, task_id=task_id, context_id=context_id)
        state, answer = None, ""
        async for ev in self._client.send_message(SendMessageRequest(message=msg)):
            kind = ev.WhichOneof("payload")
            if kind == "task":
                state, task_id, context_id = ev.task.status.state, ev.task.id, ev.task.context_id
            elif kind == "status_update":
                u = ev.status_update
                state, task_id, context_id = u.status.state, u.task_id, u.context_id
            answer = get_stream_response_text(ev) or answer
        return task_id, context_id, state, answer

    async def _ask(self, sku):
        tid, cid, state, answer = await self._send_one(
            f"What is the restock lead time for {sku}?")
        if state == TaskState.TASK_STATE_INPUT_REQUIRED:    # region pause -> answer it
            tid, cid, state, answer = await self._send_one(
                self.default_region, task_id=tid, context_id=cid)
        return {"sku": sku, "region_used": self.default_region,
                "state": TaskState.Name(state) if state is not None else None,
                "quote": answer}

    def ask_supplier(self, sku):
        fut = asyncio.run_coroutine_threadsafe(self._ask(sku), self._loop)
        return fut.result(timeout=self.timeout * 3)

In [ ]:
from shoplab.tools import Tool

a2a_bridge = A2ABridge(default_region="west")
a2a_bridge.__enter__()                                      # supplier up; torn down at the end
print("A2A peer:", a2a_bridge.card.name, "| skill:", a2a_bridge.card.skills[0].id, "| at", a2a_bridge.base_url)

def ask_supplier(sku):
    q = a2a_bridge.ask_supplier(sku)
    return {"sku": q["sku"], "region_used": q["region_used"],
            "state": q["state"], "restock_quote": q["quote"]}

a2a_tool = Tool("ask_supplier",
                "Ask the Northwind Supply peer (over A2A) for a restock ETA "
                "and wholesale quote for a Larkspur SKU (e.g. LK-1009).",
                {"type": "object",
                 "properties": {"sku": {"type": "string"}}, "required": ["sku"]},
                ask_supplier)

demo = ask_supplier("LK-1009")
print("ask_supplier('LK-1009'):", demo["state"], "| region:", demo["region_used"])
print("   ", demo["restock_quote"])

> **What you should see:** the resolved card names `Northwind Supply` and its `restock_quote` skill, and `ask_supplier('LK-1009')` comes back `TASK_STATE_COMPLETED` after the bridge silently answered the peer's region question with `west`. The quote — wholesale price, on-hand units, lead time in business days — is computed by the peer deterministically, with no model call on either side.

## Assemble the desk agent

Everything the desk needs is now on the table; the agent is what composes it. Per ticket we build a fresh toolset — the four read tools over MCP, `ask_supplier` over A2A, and the local `calc` / `escalate` / `finish` plus the risky `issue_refund` / `create_replacement` — then wrap the risky pair in a `require_approval` gate, put a `Budget` on the loop, run `check_decision` the instant the model calls `finish`, and wrap every ticket and every tool in a `Tracer` span. This is chapter 11's composition discipline — an agent assembled from parts, not one clever trick — in a leaner build: the read tools already arrive scoped over MCP, so there is no policy sub-agent here, and `check_decision` runs as a free audit beside the gate rather than wired into it. What is new is that the tools now come from two networked servers.

One realistic wrinkle: the approval gate is not a rubber stamp. Small refunds auto-approve, but anything at or above **\$200** bounces to human review — a policy the gold rules do not encode, so watch for it to override a label later.

| Piece | From | Role in this run |
|---|---|---|
| `run_agent` loop | ch02 | one model call per step; stops at `finish` |
| read tools over MCP | ch13 | `get_order` / `get_customer` / `search_policy` / `check_inventory` round-trip over stdio |
| `ask_supplier` over A2A | ch14 | a cross-org restock quote, as a tool the model may call |
| `Budget` on `on_step` | ch08 | caps each ticket at 16 calls / \$0.05 |
| `require_approval` gate | ch08 | refunds ≥ \$200 bounce to human review |
| `check_decision` verifier | ch06 | recomputes gold from `rules.decide`, no model call |
| `Tracer` + Phoenix | ch03 | a span per ticket and per tool, exported for inspection |
| `score_ticket` | ch04 | grades the run against the gold TEST split |

In [ ]:
import shoplab.llm
from shoplab.loop import run_agent
from shoplab.tools import Ledger, standard_tools, Tool
from shoplab.controls import Budget, require_approval, BudgetExceeded
from shoplab.verify import check_decision

AUTO_APPROVE_UNDER = 200.0        # small refunds auto-approve; >= this needs human review

def make_approver(ledger):
    def approve(name, args):
        if name == "issue_refund":
            amt = args.get("amount_usd", 0) or 0
            if amt < AUTO_APPROVE_UNDER:
                return True                            # auto-approve small refunds
            ledger.record("approval_escalation", tool=name, amount_usd=amt,
                          reason=f"{amt:.2f} >= auto-approve threshold {AUTO_APPROVE_UNDER}")
            return False                               # over threshold -> human review
        return True                                    # replacements pass the gate
    return approve

def make_before_tool(ticket, order, customer, state):
    """Run check_decision (ch06, no model call) the instant finish is called -- verify
    before finishing. Audit mode: record the verdict, always let finish through, grade
    afterward. The verifier is the free last line of defense, not a gate."""
    def before_tool(name, args):
        if name == "finish":
            state["verdict"] = check_decision(args, ticket, order, customer)
        return True
    return before_tool

def traced_tool(tool, tracer):                         # one Tracer span per tool call
    inner = tool.fn
    def fn(**kwargs):
        with tracer.span(tool.name, kind="tool"):
            return inner(**kwargs)
    return Tool(tool.name, tool.description, tool.params, fn, risky=tool.risky)

DESK_SYSTEM = (
    "You are the Larkspur Outfitters ops desk clearing the morning return queue. "
    "For each ticket: call get_order and get_customer to see the order line, the "
    "customer's tier and loyalty flags; call search_policy on the reason to find the "
    "governing policy; compute any dollar amount with calc. Then carry out the "
    "decision -- issue_refund when money is due, create_replacement for a warranty "
    "replacement, escalate for fraud or a flagged serial returner (or when an "
    "approval is blocked) -- and only then call finish with decision, policy_id, and "
    "refund_usd (a number, or null for replacement/deny/escalate). "
    "Rules that matter: flagged serial returners escalate (pol-fraud); damaged or "
    "defective claims over $75 need a photo or they are denied, and over $300 "
    "unevidenced they escalate; damaged-in-transit refunds add original shipping; "
    "opened change-of-mind refunds owe a 10% restocking fee unless the customer is "
    "vip; anything past the 30-day return window is denied; store-credit requests get "
    "full item value. You may call ask_supplier(sku) for a restock ETA if stock is "
    "relevant. Policy, not sympathy. Do not repeat lookups."
)

READ_FROM_MCP = ("get_order", "get_customer", "search_policy", "check_inventory")

def clear_ticket(ticket, mcp_tools, a2a_ask_tool):
    order, customer = orders[ticket["order_id"]], customers[ticket["customer_id"]]
    ledger, budget = Ledger(), Budget(max_calls=16, max_cost_usd=0.05)
    local = standard_tools(ledger)                     # local calc/escalate/finish/risky pair
    approve = make_approver(ledger)
    tools = {}
    for n in READ_FROM_MCP:
        tools[n] = mcp_tools[n]                         # <-- read tools OVER MCP
    tools["ask_supplier"] = a2a_ask_tool                # <-- peer call OVER A2A
    for n in ("calc", "escalate", "finish"):
        tools[n] = local[n]
    for n in ("issue_refund", "create_replacement"):
        tools[n] = require_approval(local[n], approve)  # risky -> approval gate
    tools = {n: traced_tool(t, tracer) for n, t in tools.items()}   # <-- Tracer spans

    state = {"verdict": None}
    before = make_before_tool(ticket, order, customer, state)
    try:
        with tracer.span("triage_ticket", kind="agent", ticket=ticket["ticket_id"]):
            r = run_agent(render(ticket), tools, system=DESK_SYSTEM, max_steps=12,
                          before_tool=before,
                          on_step=lambda step, msg: budget.charge(
                              shoplab.llm.LEDGER[-1]["cost_usd"]))
        answer, stop = r.answer, r.stop_reason
    except BudgetExceeded as e:
        answer, stop = None, f"budget:{e}"
    pred = answer if isinstance(answer, dict) else {}
    return {"ticket": ticket, "pred": pred, "stop": stop,
            "moved": ledger.entries, "budget": budget.snapshot(), "verdict": state["verdict"]}

## Clear the queue, and score it against gold

Run the assembled agent over all eight test tickets, then grade each prediction against gold with `score_ticket` and aggregate. Two things to watch: does every ticket *finish* inside its budget, and where do the predictions diverge from gold — because on a held-out split, divergences are the interesting part.

In [ ]:
from shoplab.evals import score_ticket

runs = [clear_ticket(t, mcp_tools, a2a_tool) for t in TEST]
rows = [{**r, "scores": score_ticket(r["pred"], r["ticket"]["gold"])} for r in runs]

def acc(k):
    return round(sum(x["scores"][k] for x in rows) / len(rows), 4)

print(f"{'ticket':9}{'pred':16}{'gold':16}{'pol':4}{'amt':4}{'exact':6}{'verif':6}"
      f"{'moved$':>9}{'stop':>9}")
for x in rows:
    t, p, g, s = x["ticket"], x["pred"], x["ticket"]["gold"], x["scores"]
    moved = sum(e.get("amount_usd", 0) for e in x["moved"] if e["kind"] == "issue_refund")
    verif = x["verdict"]["agrees"] if x["verdict"] else None
    print(f"{t['ticket_id']:9}{str(p.get('decision')):16}{g['decision']:16}"
          f"{str(s['policy_correct'])[0]:4}{str(s['amount_correct'])[0]:4}"
          f"{str(s['exact'])[0]:6}{str(verif)[0]:6}{moved:>9.2f}{x['stop']:>9}")

print(f"\nn={len(rows)}  decision {acc('decision_correct'):.3f}  policy {acc('policy_correct'):.3f}"
      f"  amount {acc('amount_correct'):.3f}  exact {acc('exact'):.3f}")

for x in rows:                                         # approval gate side effects
    ap = [e for e in x["moved"] if e["kind"] == "approval_escalation"]
    if ap:
        print(f"gate: {x['ticket']['ticket_id']} refund ${ap[0]['amount_usd']:.2f} "
              f">= ${AUTO_APPROVE_UNDER} -> blocked, sent to human review")

us = shoplab.llm.usage_summary()
print(f"cost: {us['calls']} calls, {us['prompt_tokens']}+{us['completion_tokens']} tokens, "
      f"${us['cost_usd']:.6f}  (cached reruns bill $0)")

> **What you should see:** all eight tickets end at `finish`, inside the 16-call / \$0.05 per-ticket budget, and the aggregate lands near `exact 0.75` — roughly six of eight matching gold on decision, policy, and amount to the cent. The two misses are worth reading, not smoothing over. On one, the small model labels an opened-item refund `approve_refund` where the rule says `partial_refund` — yet it still computed the fee-reduced amount correctly. On another, the \$200 approval gate turns a would-be refund into an `escalate`: a policy choice overriding the label, exactly as designed. Total cost is a couple of cents live, `$0` cached.

## Read the trace

Every ticket and every tool ran inside a `Tracer` span, and Phoenix auto-captured every model call underneath. Print the span tree for one ticket to see the shape of a single triage, then the tool tally across the whole queue, then confirm the spans reached Phoenix. `TKT-2237` is the instructive one — it is where the approval gate fired.

In [ ]:
from collections import Counter

one = "TKT-2237"
ids = {s.span_id for s in tracer.spans if s.attributes.get("ticket") == one}
subtree = [s for s in tracer.spans if s.span_id in ids or s.parent_id in ids]

depth = {}
for s in subtree:                                      # one line per span, indented by depth
    d = depth[s.span_id] = depth.get(s.parent_id, -1) + 1
    dur = "..." if s.duration_ms is None else f"{s.duration_ms:.0f} ms"
    print("  " * d + f"{s.name} [{s.kind}] {dur}")

print("\nspan tally across the queue:")
for (kind, name), c in sorted(Counter((s.kind, s.name) for s in tracer.spans).items()):
    print(f"  {kind:6} {name:16} x{c}")

print("\nexport to Phoenix:", export_phoenix(tracer), "| spans exported:", len(tracer.spans))

> **What you should see:** one `triage_ticket` agent span with its tool spans nested underneath — `get_order`, `get_customer`, `search_policy`, a `calc`, the gated `issue_refund`, then `escalate` and `finish` — the whole life of one ticket as a tree. The tally shows eight `triage_ticket` spans (one per ticket) and the tool mix across the queue, and `export_phoenix` returns `200`: the same spans are now browsable in Phoenix under today's project, beside the LLM spans it captured on its own.

## Teardown

Both servers were started with a manual `__enter__`, so we stop them explicitly now that the run is done. Closing the MCP bridge lets its stdio context unwind, which reaps the server subprocess; closing the A2A bridge signals its uvicorn thread and joins it. Then we check that nothing is left behind — no bound supplier port. Phoenix on 6006 is left running on purpose: `obs` finds and reuses it across notebooks.

In [ ]:
import socket

a2a_port = int(a2a_bridge.base_url.rsplit(":", 1)[1])
mcp_bridge.__exit__(None, None, None)                       # stdio context unwinds; subprocess reaped
a2a_bridge.__exit__(None, None, None)                       # uvicorn thread signalled and joined

def port_bound(p):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", p)) == 0

print("MCP bridge closed; A2A bridge closed.")
print("A2A port", a2a_port, "still bound:", port_bound(a2a_port))
print("(Phoenix on 6006 stays up on purpose -- obs.py reuses it across notebooks.)")

> **What you should see:** both bridges report closed, and the supplier port comes back `still bound: False` — no orphan subprocess, no leaked port. The only server still up is Phoenix, deliberately.

## What you built

The finale had no new idea in it, and that is the point. Every capability the desk used tonight was built, by hand, earlier in the course; the capstone only wired them together and pointed them at tickets none of them had seen.

| Capability | Chapter | In this capstone |
|---|---|---|
| The model boundary + cost ledger | 01 | every triage call runs through `shoplab.llm.complete`; `usage_summary` totals the run |
| The tool-calling loop | 02 | `run_agent`, unchanged, drives all eight tickets |
| Tracing | 03 | a `Tracer` span per ticket and tool, exported to Phoenix |
| Evaluation vs gold | 04 | `score_ticket` grades against the held-out TEST split |
| Verification | 06 | `check_decision` audits each `finish` against `rules.decide` |
| Budgets + approval gates | 08 | a per-ticket `Budget`; refunds ≥ \$200 gated to human review |
| Composition over one trick | 11 | the per-ticket agent is a plain loop wrapped in the guardrails you built — assembled, not new |
| Tools over MCP | 13 | four read tools reach the loop over a stdio `ClientSession` |
| A2A peer call | 14 | `ask_supplier` consults Northwind Supply over the wire |

The honest read of the score is the lesson: six of eight exact, two explicable divergences — one small-model label slip, one deliberate policy override — on tickets held out from the whole course. A capable agent is not one clever trick; it is several plain ones, composed, measured, and kept inside guardrails you can name. This is the last numbered chapter. The appendices take the same ops desk into other frameworks — Pydantic AI, ADK, LlamaIndex, Agno, and the isolated-venv trio — so you can see the identical brief expressed several more ways, and judge each packaging against the machinery you now own.

## Recap

| Concept | One-liner |
|---|---|
| The capstone | the morning queue cleared once more, but every subsystem is the real one you built. |
| Read tools over MCP | `get_order` / `get_customer` / `search_policy` / `check_inventory` round-trip through a live `ClientSession`. |
| Peer call over A2A | `ask_supplier` reaches Northwind Supply over the wire, region pause and all, as one sync tool. |
| The composed agent | `run_agent` + `Budget` + a `require_approval` gate + a `check_decision` audit + `Tracer`, assembled per ticket. |
| Held-out evaluation | `score_ticket` against the gold TEST split — eight tickets no chapter fit anything to. |
| Honest result | around six of eight exact; the misses are a label slip and a gate override, both explicable. |
| Measured cost | `usage_summary` totals the run at a couple of cents live, `$0` cached. |
| Traced end to end | a span per ticket and tool, exported to Phoenix beside the auto-captured LLM spans. |
| Clean teardown | both servers stopped in-notebook — no orphan subprocess, no bound port. |

## Exercises

1. **Move one tool back across the wire.** In `clear_ticket`, source `check_inventory` from `standard_tools(ledger)` (local) instead of from `mcp_tools`, leaving the other three read tools over MCP, and re-run the queue. Do the decisions or the cost change at all? They should not — which is the whole point of chapter 13's adapter: the loop cannot tell a local tool from a remote one. What *does* change is the trace and the process tree; find where, and say why that difference is the only one that matters operationally.
2. **Add a second A2A peer.** Stand up a second supplier (copy `a2a_servers/supplier_agent.py`, change the card `name` and the wholesale multiplier) on its own port, wrap it as `ask_supplier_b`, and give the agent both. On a low-stock ticket, does the model consult one peer, both, or neither? What in the system prompt or the tool descriptions would make it comparison-shop — and is comparison-shopping something you actually want it doing unprompted? (Cost-flag: a few live model calls on the first run.)
3. **Is the held-out score honest?** Run the same assembled agent over the *dev* split (`load_tickets()["dev"]`, `TKT-2221`–`TKT-2232`) and compare its `exact_acc` to this test result and to the dev numbers chapter 04 recorded. If dev and test land close, the score generalizes; if test is much easier or harder, ask whether eight tickets is enough to trust — and what you would measure to decide. (Cost-flag: twelve uncached triages on the first run.)